In [ ]:
import pandas as pd
import numpy as np
import os
import datetime as dt
import pathlib2 as pl2
import warnings
warnings.filterwarnings("ignore")

import inspect
from functools import partial
from multiprocessing import Pool
import pyarrow.parquet as pq
from pathlib import Path

In [ ]:
trade_data_path_1 = "<local-path>"

In [ ]:
completed_file_path = "<local-path>"

completed_file_list_1 = os.listdir(completed_file_path)

test_year_list = [2025,2026]

r_list = []
for completed_file in completed_file_list_1:
    for y in test_year_list:
        
        new_c_path = completed_file_path + completed_file + "/Final Summary/" + str(y)

        try:
            c_file_list = os.listdir(new_c_path)
            if len(c_file_list) == 2:
                pass
            else:
                r_list.append(completed_file)
        
        except:
            r_list.append(completed_file)
            
        


In [ ]:
missing_list_1 = list(set(r_list))
missing_list_1

In [ ]:
files = os.listdir(trade_data_path_1)
# files.remove("77 strategy")
# files.remove("Long Term Pull Back")
# files.remove("William Vix Fix")
# files.remove("William Vix Fix (Normal)")

# files = ['123_pattern']

# files = files
files

In [ ]:
files = list(set(list(set(files) - set(completed_file_list_1)) + missing_list_1))
files

In [ ]:
# Major_pairs_list = ["AUD/CHF","AUD/JPY","AUD/NZD","AUD/USD","CAD/CHF","CAD/JPY","CHF/JPY","EUR/AUD","EUR/CAD",
#                    "EUR/CHF","EUR/GBP","EUR/JPY","EUR/NZD","EUR/USD","GBP/AUD","GBP/CAD","GBP/CHF","GBP/JPY",
#                    "GBP/NZD","GBP/USD","NZD/CAD","NZD/CHF","NZD/JPY","NZD/USD","USD/CAD","USD/CHF","USD/JPY","AUD/CAD"]

# Exotic_pairs_list = ['EUR/NOK','EUR/SEK','USD/MXN','USD/ZAR','USD/CNH','USD/NOK','ZAR/JPY','USD/SEK']

# Commodities_pairs_list = [ "XBR/USD","XTI/USD","XNG/USD","XCU/USD",'XAU/USD','XAG/USD']

# ETF_pairs_list = ['AUS/USD', 'CHN/USD', 'ESP/USD', 'EUS/USD', 'FRA/USD', 'GER/USD',
#                    'HKG/USD', 'JPN/USD', 'NAS/USD', 'SPX/USD', 'UK1/USD', 'US3/USD']
               

In [ ]:
def Func(it,iteration_list): 
    
    
    import pandas as pd
    import numpy as np
    import os
    import datetime as dt
    import pathlib2 as pl2
    import warnings
    warnings.filterwarnings("ignore")
    
    import pyarrow.parquet as pq
    from pathlib import Path

    
    

    
    i = iteration_list[it]

    
#     file = i[0]
    file = i
    
    
    print("Current strategy:",file)
    
    # ------------ Backtesting Results path ---------------- #
    
    strategy_name = file
    initial_path = "<local-path>" + str(strategy_name) + "/"
        
    initial_path_testing = "<local-path>" + str(strategy_name) + "/"
    
    #     bt_algo_path = initial_path + "BT/"
    
    export_path =  initial_path_testing + "Final Summary/"    
    
    path = initial_path + "Final Summary/"
    trade_data_path = initial_path + "Trade Level Data/"
    
    # ------------------------------------------------------ #
    
    
    # ------------ Code to concat multiples file to one using list Comprehension ------------------ #
    
    try:
    
        print("Collating Files for : ", strategy_name)
        
         # Assuming you have a list of Parquet file paths (parquet_file_paths) using Path.glob
        parquet_file_paths = Path(trade_data_path).glob('*.parquet')
    
        # Using a generator expression to read Parquet files into DataFrames
        dfs = (pq.read_table(p).to_pandas() for p in parquet_file_paths)
    
        # Concatenating the DataFrames into a single DataFrame
        all_bt_files = pd.concat(dfs, ignore_index=True)
    
        print("Collating completed for : ", strategy_name)
    
    #         # ---------- Folder path where all files is located -------------------- #
    #         file_path = pl2.Path(trade_data_path)
    #         # ---------------------------------------------------------------------- #
    
    #         # ---------- Reading all Files using list Comprehension --------------- #
    #         dfs = (pd.read_csv(p, encoding='utf8') for p in file_path.glob('*.csv'))
    
    #         # -------- Now concating all files to single file using concat function ------------- #
    #         all_bt_files = pd.concat(dfs,ignore_index = True)
    #         # ----------------------------------------------------------------- #
        
    except Exception as e:
        print(e)
        print("No Trades found for : ", strategy_name)
        all_bt_files = pd.DataFrame()
    #         continue
    
    
    
    # ------------ Filtering BT data from 2010 ----------------------------------- #
    
    try:
    
        all_bt_files['date'] = pd.to_datetime(all_bt_files['date'])
        all_bt_files = all_bt_files[all_bt_files['date'] >= dt.datetime(2010,1,1,0,0,0)].reset_index(drop = True)
    
    
        all_bt_files['Single trade flag'] = np.where(all_bt_files['date'] < all_bt_files['Closing Date'].shift(1),1,0)
    
        all_bt_files = all_bt_files[all_bt_files['Single trade flag'] == 0].reset_index(drop = True)
    
    
    
        # --------------------------------------------------------------------------- #
    
        try:
            os.mkdir(initial_path_testing)
            os.mkdir(export_path)
        except:
            print("Folder Already Created")
    
    
    
        test_year_list = [2025]
    
    
        for test_year in test_year_list:
    
            
            BT_data = all_bt_files[(all_bt_files['date'] < dt.datetime(2025,10,1,0,0,0)) &
                         (all_bt_files['Closing Date'] < dt.datetime(2025,10,1,0,0,0))].reset_index(drop = True)
    
            FT_data = all_bt_files[(all_bt_files['date'] >= dt.datetime(2025,10,1,0,0,0)) &
                         (all_bt_files['Closing Date'] >= dt.datetime(2025,10,1,0,0,0))].reset_index(drop = True)
    
    
    
    
#             BT_data = all_bt_files[(all_bt_files['EntryDate Year'] < test_year) &
#                          (all_bt_files['ClosingDate Year'] < test_year)].reset_index(drop = True)
    
#             FT_data = all_bt_files[(all_bt_files['EntryDate Year'] == test_year) &
#                          (all_bt_files['ClosingDate Year'] == test_year)].reset_index(drop = True)
    
    
    
            train_test_path = export_path + str(test_year) + "/"
    
            try:
                os.mkdir(train_test_path)
            except:
                print("Folder Already Created")
                
                
            BT_data['date'] = pd.to_datetime(BT_data['date'])
            BT_data['Closing Date'] = pd.to_datetime(BT_data['Closing Date'])
            BT_data['Year'] = BT_data['date'].dt.year
    
    
            #---------------------------#
            BT_data['EntryDate Year'] = BT_data['date'].dt.year
            BT_data['EntryDate Month'] = BT_data['date'].dt.month
    
            BT_data['ClosingDate Year'] = BT_data['Closing Date'].dt.year
            BT_data['ClosingDate Month'] = BT_data['Closing Date'].dt.month
    
    
            #---calculation of ppm by trade opening date ------#
    
    
            data_pos_1 = pd.DataFrame()
            data_pos_1 = BT_data.groupby(by = ['Strategy', 'Currency', 'Condition', 'Timeframe',
                                               'parameter_1', 'parameter_2', 'parameter_3',
                                                              'parameter_4', 'parameter_5', 'parameter_6',
                                               'EntryDate Year','EntryDate Month'])["Profit_Loss"].sum()
            data_pos_1 = pd.DataFrame(data_pos_1)
            data_pos_1.reset_index(inplace=True)
            data_pos_1['Positive_months by Trade Open Date'] = np.where(data_pos_1["Profit_Loss"] > 0,1,0)
    
    
            #---calculation of ppm by trade closing date ------#
            data_pos_2 = pd.DataFrame()
            data_pos_2 = BT_data.groupby(by = ['Strategy', 'Currency', 'Condition', 'Timeframe',
                                               'parameter_1', 'parameter_2', 'parameter_3',
                                                              'parameter_4', 'parameter_5', 'parameter_6',
                                                'ClosingDate Year','ClosingDate Month'])["Profit_Loss"].sum()
            data_pos_2 = pd.DataFrame(data_pos_2)
            data_pos_2.reset_index(inplace=True)
            data_pos_2['Positive_months by Trade Close Date'] = np.where(data_pos_2["Profit_Loss"] > 0,1,0)
    
    
            # ----------------------- creating Summary ---------------------- #    
    
            # ------------------- BackTesting Data ---------------------- #  
    
    
    #         Back_testing_data = pd.DataFrame(BT_data.groupby(['Strategy','Currency','Condition','Timeframe','parameter_1','parameter_2','parameter_3','parameter_4',
    #                                                                 'parameter_5','parameter_6']).agg(
    #                                                                 Net_P_L=('Profit_Loss', 'sum'),
    #                                                                 Total_Trades=('Profit_Loss', 'count'),
    #                                                                 Profitable_Trades = ('Good_Bad_Flag','sum'))).reset_index()
    
    
            Back_testing_data_1 = pd.DataFrame(BT_data.groupby(['Strategy', 'Currency', 'Condition', 'Timeframe',
                                                              'parameter_1', 'parameter_2', 'parameter_3',
                                                              'parameter_4', 'parameter_5', 'parameter_6']).agg(
                                                                Net_P_L=('Profit_Loss', 'sum'),
                                                                Total_Trades=('Profit_Loss', 'count'),
                                                                Profitable_Trades=('Good_Bad_Flag', 'sum'),
                                                                Positive_Avg_Profit=('Profit_Loss', lambda x: x[x > 0].mean()),
                                                                Negative_Avg_Profit=('Profit_Loss', lambda x: x[x < 0].mean()),
                                                                Positive_Trades_Sum=('Profit_Loss', lambda x: x[x > 0].sum()),
                                                                Negative_Trades_Sum=('Profit_Loss', lambda x: x[x < 0].sum()))).reset_index()


            data_pos_2_summary = pd.DataFrame(data_pos_2.groupby(['Strategy', 'Currency', 'Condition', 'Timeframe',
                                                              'parameter_1', 'parameter_2', 'parameter_3',
                                                              'parameter_4', 'parameter_5', 'parameter_6']).agg(
                                                                Total_positive_month_by_close_date=('Positive_months by Trade Close Date', 'sum'),
                                                                Total_month_by_close_date=('Positive_months by Trade Close Date', 'count'))).reset_index()

            data_pos_2_summary['% Positive Months by Trade Close Date'] = data_pos_2_summary['Total_positive_month_by_close_date']/data_pos_2_summary['Total_month_by_close_date']
    
            Back_testing_data = Back_testing_data_1.merge(data_pos_2_summary, on = ['Strategy', 'Currency', 'Condition', 'Timeframe',
                                                              'parameter_1', 'parameter_2', 'parameter_3',
                                                              'parameter_4', 'parameter_5', 'parameter_6'], how = 'left')
            
    
    
            # -------------------------------------------------------------- #
    
            # -------------------Fwd Testing Data ---------------------- #
    
            Fwd_testing_data = pd.DataFrame(FT_data.groupby(['Strategy','Currency','Condition','Timeframe','parameter_1','parameter_2','parameter_3','parameter_4',
                                                                    'parameter_5','parameter_6']).agg(
                                                                Net_P_L=('Profit_Loss', 'sum'),
                                                                Total_Trades=('Profit_Loss', 'count'),
                                                                Profitable_Trades=('Good_Bad_Flag', 'sum'),
                                                                Positive_Avg_Profit=('Profit_Loss', lambda x: x[x > 0].mean()),
                                                                Negative_Avg_Profit=('Profit_Loss', lambda x: x[x < 0].mean()),
                                                                Positive_Trades_Sum=('Profit_Loss', lambda x: x[x > 0].sum()),
                                                                Negative_Trades_Sum=('Profit_Loss', lambda x: x[x < 0].sum()))).reset_index()
    
    
            Fwd_testing_data['Accuracy'] = Fwd_testing_data['Profitable_Trades']/Fwd_testing_data['Total_Trades']
            Fwd_testing_data['PPT'] = Fwd_testing_data['Net_P_L']/Fwd_testing_data['Total_Trades']
    
    
    
            # ---------------------------------------------------------------- #
    
            Back_testing_data.to_excel(train_test_path + " Train data summary.xlsx",index = False)
    
            Fwd_testing_data.to_excel(train_test_path + " Test data summary.xlsx",index = False)
    
    
    
    
    
    
    
    except Exception as e:
        
        print(e,"Error")

In [ ]:

cpu_count = 45

def parallel_task(func, iteration_list,cpus):

    with open(f'./tmp_func.py', 'w') as file:
        file.write(inspect.getsource(func).replace(func.__name__, "task"))
        
    from tmp_func import task

    if __name__ == '__main__':
        func = partial(task)
        pool = Pool(processes=cpus)
        res = pool.starmap(func,([i,iteration_list] for i in range(len(iteration_list))))
        pool.close()
        pool.join()
        return res
    else:
        raise "Not in Jupyter Notebook"
# iterable = [i for i in range(len(iteration_list))]
outputs = parallel_task(Func,files,cpus = cpu_count)
